# Stage 3, step 1: the BioLangFusion benchmark datasets

Research plan item 1 (Priority 1), Phase A, needs BioLangFusion reproduced *exactly*: its own
encoders, its codon grid, its three fusion heads and TextCNN head, on its five published
benchmarks, with the published splits and recipe (Gantt 17). Nothing in that reproduction is
meaningful unless the data going into it is the data the paper used.

This notebook is the data step of that build. It answers four questions, in order:

1. Which of the five benchmarks are in hand, and which are not?
2. Does each file carry what the reproduction needs: one coding sequence per row, a label, and
   a published train/validation/test assignment?
3. BioLangFusion reports how many sequences it *used* from each benchmark, but not the
   preprocessing rule that produced those counts. Can the rule be recovered?
4. What is wrong with the published data that the reproduction has to report rather than hide?

No model is loaded and no embedding is computed here; that is
step 2, `02_token_embeddings.ipynb`. The datasets this notebook checks are registered in
[`shared_code.datasets`](../shared_code/datasets.py), so every later Stage 3 and Stage 4
notebook loads them the same way. See [the Stage 3 plan](README.md) for the surrounding steps.


---

## Setup

### Imports

The audit uses pandas and the project's dataset registry; no encoder, PyTorch, or network
access is needed. The path block finds `Code/shared_code/` from this notebook's folder,
`Code/stage3/`, so the notebook runs with its own folder as the working directory.


In [1]:
# Standard library
import sys
from pathlib import Path

# Data handling
import pandas as pd

# Shared project code in Code/shared_code; this notebook lives in Code/stage3
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets

pd.set_option("display.width", 120)
BENCHMARKS = datasets.BLF_BENCHMARKS
print("Registered BioLangFusion benchmarks:", ", ".join(BENCHMARKS))
print("Length rule applied by blf_benchmark_rows:", datasets.BLF_LENGTH_LIMIT, "nucleotides")

Registered BioLangFusion benchmarks: fungal_expression, ecoli_proteins, mrna_stability, cov_vac_degradation
Length rule applied by blf_benchmark_rows: 999 nucleotides


---

## The five benchmarks BioLangFusion reports

BioLangFusion fine-tunes three frozen encoders (Nucleotide Transformer v2 100M multi-species,
RNA-FM, and ESM-2 8M) on five CodonBERT benchmarks, and reports, for each one, the score of
each single encoder and of its best fusion head. Those numbers are the targets Phase A has to
hit. The counts below are the number of sequences the paper states it used, which is smaller
than the files for the three longest benchmarks.


In [2]:
# BioLangFusion's reported results and the number of sequences it used, from its Table 1.
# "NT" is Nucleotide Transformer v2 100M multi-species. Scores are Spearman correlation except
# E. coli, which is accuracy. These are the targets Phase A has to hit.
PUBLISHED = pd.DataFrame(
    [
        ("Fungal expression",   "fungal_expression",   "Spearman",  3138, 0.804, 0.767, 0.695, 0.824, "gated MIL"),
        ("E. coli proteins",    "ecoli_proteins",      "accuracy",  4450, 41.3,  52.9,  49.0,  53.9,  "cross-attention"),
        ("mRNA stability",      "mrna_stability",      "Spearman", 23929, 0.530, 0.553, 0.539, 0.563, "gated MIL"),
        ("Ab1 antibody mRNA",   None,                  "Spearman",   723, 0.732, 0.743, 0.711, 0.765, "cross-attention"),
        ("CoV-Vac degradation", "cov_vac_degradation", "Spearman",  2400, 0.780, 0.841, 0.806, 0.864, "gated MIL"),
    ],
    columns=["benchmark", "registry key", "metric", "used", "NT", "RNA-FM", "ESM-2 8M", "best fused", "best head"],
)
PUBLISHED = PUBLISHED.assign(**{
    "best single": PUBLISHED[["NT", "RNA-FM", "ESM-2 8M"]].max(axis=1),
})
PUBLISHED = PUBLISHED.assign(**{
    "fusion gain": (PUBLISHED["best fused"] - PUBLISHED["best single"]).round(3),
    "in hand": PUBLISHED["registry key"].notna().map({True: "yes", False: "NO"}),
})
PUBLISHED.set_index("benchmark")[
    ["metric", "used", "in hand", "NT", "RNA-FM", "ESM-2 8M", "best single", "best fused", "best head", "fusion gain"]
]

,metric,used,in hand,NT,RNA-FM,ESM-2 8M,best single,best fused,best head,fusion gain
benchmark,,,,,,,,,,
Fungal expression,Spearman,3138,yes,0.804,0.767,0.695,0.804,0.824,gated MIL,0.020
E. coli proteins,accuracy,4450,yes,41.300,52.900,49.000,52.900,53.900,cross-attention,1.000
mRNA stability,Spearman,23929,yes,0.530,0.553,0.539,0.553,0.563,gated MIL,0.010
Ab1 antibody mRNA,Spearman,723,NO,0.732,0.743,0.711,0.743,0.765,cross-attention,0.022
CoV-Vac degradation,Spearman,2400,yes,0.780,0.841,0.806,0.841,0.864,gated MIL,0.023


The gains to reproduce are small: 0.010 to 0.023 on the four
regression benchmarks, and 1.0 accuracy point on *E. coli*. The paper reports no variance, so
Phase A adds seeds and paired differences (Overleaf Section 12.2, item 1).

**Ab1 is not in hand.** Four benchmark files are available from the CodonBERT
fine-tuning benchmark recorded in the registry; the antibody set, which BioLangFusion
describes as 723 heavy-chain sequences of up to 1,203 nucleotides, is not among them and was
not located in that release. Phase A therefore covers four of the five benchmarks for now, and
Ab1 stays an open item: either the file is obtained from the CodonBERT authors, or the
reproduction is reported on four benchmarks with the omission stated. This is the first
question for the October 20 meeting.


---

## What is in each file

The four available benchmarks share one schema: `Sequence`, `Value`, `Dataset`, and the
published `Split`. The audit below reports the file as distributed, before any preprocessing.


In [3]:
# Each file as distributed, before any preprocessing.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    frame = pd.read_csv(datasets.DEFAULT_DATA_DIR / spec.path)
    seq = frame[spec.sequence_column].astype(str).str.strip().str.upper()
    value = frame[spec.label_column]
    rows.append({
        "benchmark": spec.label,
        "file": spec.path,
        "rows": len(frame),
        "columns": ", ".join(frame.columns),
        "alphabet": "".join(sorted(set("".join(seq.head(500))))),
        "length min": int(seq.str.len().min()),
        "length median": int(seq.str.len().median()),
        "length max": int(seq.str.len().max()),
        "task": spec.task,
        "label min": round(float(value.min()), 3),
        "label max": round(float(value.max()), 3),
        "distinct labels": int(value.nunique()),
    })
AUDIT = pd.DataFrame(rows).set_index("benchmark")
AUDIT

,file,rows,columns,alphabet,length min,length median,length max,task,label min,label max,distinct labels
benchmark,,,,,,,,,,,
Fungal expression,Fungal_expression.csv,7089,"Sequence, Value, Dataset, Split",ACGU,150,1086,3063,regression,-8.977,5.14,5134
"E. coli protein expression (low, medium, high)",E.Coli_proteins.csv,6348,"Sequence, Value, Dataset, Split",ACGU,171,747,3000,classification,0.000,2.00,3
mRNA stability,mRNA_Stability.csv,65356,"Sequence, Value, Dataset, Split",ACGU,3,1239,3066,regression,-3.109,2.89,65192
CoV-Vac degradation,CoV_Vaccine_Degradation.csv,2400,"Sequence, Value, Dataset, Split",ACGU,81,81,81,regression,-7.154,6.51,2378


Every sequence is RNA letters, and every length is a multiple of three,
so each row is a coding sequence read in frame. The label is continuous on four benchmarks and
a three-class integer on *E. coli*, which is why that benchmark is scored by accuracy and the
others by Spearman correlation.

The *CoV-Vac* rows are 81 nucleotides and fixed-length: they are fragments of a vaccine
construct, not whole coding sequences. That matters for the codon grid, and
[Sequence checks](#Sequence-checks) returns to it.


---

## Recovering BioLangFusion's preprocessing

BioLangFusion states how many sequences it used from each benchmark but not how it chose them,
and the files hold more rows than that. Its encoders have a 1,000-token input limit, which
suggests a length cut rather than a sample. If that is the rule, one cut applied to all four
files should reproduce all four published counts at once; a sampling rule would not.

The sweep below applies every length limit from 900 to 1,100 nucleotides and reports which
ones reproduce all four counts exactly.


In [4]:
# Does one length cut reproduce all four published counts at once?
target = PUBLISHED.dropna(subset=["registry key"]).set_index("registry key")["used"].to_dict()
matching = [
    limit for limit in range(900, 1101)
    if all(len(datasets.blf_benchmark_rows(k, length_limit=limit)) == target[k] for k in BENCHMARKS)
]
print("Length limits reproducing all four published counts:", matching)

SWEEP = pd.DataFrame([
    {
        "benchmark": datasets.DATASETS[k].label,
        "rows in file": len(datasets.blf_benchmark_rows(k, length_limit=10**9)),
        "kept at 999 nt": len(datasets.blf_benchmark_rows(k)),
        "published used": target[k],
        "matches": len(datasets.blf_benchmark_rows(k)) == target[k],
    }
    for k in BENCHMARKS
]).set_index("benchmark")
SWEEP

Length limits reproducing all four published counts: [999, 1000, 1001]


,rows in file,kept at 999 nt,published used,matches
benchmark,,,,
Fungal expression,7089,3138,3138,True
"E. coli protein expression (low, medium, high)",6348,4450,4450,True
mRNA stability,65356,23929,23929,True
CoV-Vac degradation,2400,2400,2400,True


One rule reproduces all four counts exactly: **keep sequences of at
most 999 nucleotides, and drop the longer ones rather than truncating them.** Limits of 1,000
and 1,001 give the same result only because every length is a multiple of three, so 999 is the
largest length that can occur below 1,002; the rule is a 1,000-token budget, one start token
plus up to 999 nucleotides.

Reproducing four independent counts with one rule is strong evidence, but it is still an
inference from counts, not a statement from the paper. It is recorded as a deviation
(Section [Deviations](#Deviations-recorded-for-Phase-A)) and
[`blf_benchmark_rows`](../shared_code/datasets.py) applies it, so every later notebook uses
the same rule and any correction changes one place.

Dropping rather than truncating also matters scientifically: a truncated coding sequence loses
its stop codon and its 3' end, and the three streams would no longer end on the same codon.

One thing the rule does **not** explain. BioLangFusion's Appendix A.2 reports 41,123 *raw*
mRNA stability sequences against this file's 65,356, so the raw counts disagree even though
the used counts agree exactly. The file in hand is therefore a different release of the
benchmark from the one the paper started with, and the agreement at 23,929 says the two
releases share the same short-sequence subset, not that they are the same file. The README
already lists this as an open question about the stability data; it stays open, and it is the
reason Phase A reports the row count it actually used beside every score.

**The paper says it did something else.** Appendix A.2 states: "To ensure compatibility with
pretrained language models, most of which support input lengths up to 1000 tokens, we truncate
RNA sequences exceeding this length." Truncation keeps every row, so it cannot produce the
counts in the paper's own Table 2, where fungal expression falls from 7,056 to 3,138 and mRNA
stability from 41,123 to 23,929. Dropping reproduces those counts exactly; truncating cannot
reduce them at all. The prose and the table disagree, and the table agrees with the files.

One row of that table points the other way, and it is the one benchmark we do not have. Ab1 has
a maximum length of 1,203 nucleotides and goes from 723 raw to 723 used: nothing was dropped
from it, which is what truncation would do and what a 999-nucleotide cut would not. So either
Ab1 was treated differently from the four CodonBERT sets, or the rule is not uniform. That
cannot be settled without the Ab1 file, and it is worth raising with Mina alongside the request
for the data.

---

## Published splits, and how much the test split reuses

Phase A must keep each benchmark's published split. The table below reports that split after
the 999-nucleotide rule, and then asks a question the published counts do not answer: how many
test sequences the model has already seen in training.


In [5]:
# The published split after the 999-nucleotide rule, and how much of each test split is reused.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    frame = datasets.blf_benchmark_rows(key)
    split = frame[spec.split_column]
    train_seqs = set(frame.loc[split == "train", "sequence"])
    val_seqs = set(frame.loc[split == "val", "sequence"])
    test = frame[split == "test"]
    test_seqs = set(test["sequence"])
    repeated = frame.groupby("sequence")[spec.label_column]
    sizes, spreads = repeated.size(), repeated.max() - repeated.min()
    rows.append({
        "benchmark": spec.label,
        "train": int((split == "train").sum()),
        "val": int((split == "val").sum()),
        "test": int((split == "test").sum()),
        "distinct sequences": int(frame["sequence"].nunique()),
        "sequences in >1 split": int((frame.groupby("sequence")[spec.split_column].nunique() > 1).sum()),
        "test rows seen in train": int(test["sequence"].isin(train_seqs).sum()),
        "test sequences unseen": len(test_seqs - train_seqs - val_seqs),
        "repeats with differing labels": int((spreads[sizes > 1] > 0).sum()),
    })
SPLITS = pd.DataFrame(rows).set_index("benchmark")
SPLITS

,train,val,test,distinct sequences,sequences in >1 split,test rows seen in train,test sequences unseen,repeats with differing labels
benchmark,,,,,,,,
Fungal expression,2153,555,430,3131,0,0,430,6
"E. coli protein expression (low, medium, high)",3010,728,712,4450,0,0,712,0
mRNA stability,16742,3608,3579,11303,3178,2444,985,4549
CoV-Vac degradation,1600,400,400,2400,0,0,400,0


Three benchmarks are clean: no sequence of *fungal expression*,
*E. coli*, or *CoV-Vac* appears in more than one split.

**The mRNA stability benchmark is not.** Its 23,929 rows hold only 11,303 distinct sequences,
2,444 of its 3,579 test rows (68%) carry a sequence that also appears in training, and only
985 of its 3,021 distinct test sequences are unseen. Worse, 4,549 of the 4,579 repeated
sequences carry *different* labels, with a median spread of 0.94 on a label range of about
six: the same coding sequence is measured more than once, presumably in different cell lines
or assays, and the benchmark keeps each measurement as its own row without keeping them on the
same side of the split.

This does not stop the reproduction, and Phase A still reports the published split, because
reproducing the published number is the point. It does change what the number means: a large
part of the reported mRNA stability performance can be reached by recognizing a training
sequence and recalling its label. Stage 1 already met this as the 295-sequence gap on its own
grouped split; this quantifies it on the published one.

Phase A therefore reports mRNA stability twice, as the Overleaf reproduction protocol already
requires: once on the published split, to match the paper, and once on the sequence-grouped
split, to show what survives when the overlap is removed. The gap between them is a result in
its own right.


## Sequence checks

Three properties decide how a row reaches the codon grid: its alphabet, whether its length is
a multiple of three, and whether it starts with a start codon. The Stage 1 filters
([`is_retained`](../shared_code/datasets.py)) require all three plus a five-residue
translation, and are deliberately *not* applied here.


In [6]:
# Properties that decide how a row reaches the codon grid.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    seq = datasets.blf_benchmark_rows(key)["sequence"]
    rows.append({
        "benchmark": spec.label,
        "kept rows": len(seq),
        "RNA letters only": bool(set("".join(seq)) <= set("ACGU")),
        "length a multiple of 3": f"{(seq.str.len() % 3 == 0).mean():.3f}",
        "starts with AUG": f"{(seq.str[:3] == 'AUG').mean():.3f}",
        "dropped by the Stage 1 filters": f"{1 - seq.map(datasets.is_retained).mean():.3f}",
    })
CHECKS = pd.DataFrame(rows).set_index("benchmark")
CHECKS

,kept rows,RNA letters only,length a multiple of 3,starts with AUG,dropped by the Stage 1 filters
benchmark,,,,,
Fungal expression,3138,True,1.000,0.932,0.068
"E. coli protein expression (low, medium, high)",4450,True,1.000,1.000,0.000
mRNA stability,23929,True,1.000,0.978,0.022
CoV-Vac degradation,2400,True,1.000,0.016,0.989


The reason not to apply the Stage 1 filters is in the last column: only
1.6% of the *CoV-Vac* fragments begin with AUG, so the Stage 1 rule would discard a benchmark
BioLangFusion reports. *Fungal expression* would lose 6.8% of its rows and mRNA stability 2.2%
for the same reason.

So the reproduction takes each file as the benchmark defines it. What a sequence without a
start codon contributes to the protein stream is a decision for the pipeline itself, in
step 3, `03_blf_build_check.ipynb`, not a reason to drop the row here. The three plausible
treatments (translate from the first base regardless, translate the longest open reading
frame, or supply no protein stream and let the fusion head see a masked modality) give
different protein inputs for the same nucleotide input, and the choice is recorded as a
deviation because BioLangFusion does not state one.


---

## Deviations recorded for Phase A

Phase A's success criterion is matching the published scores *or recording each deviation*.
Four are known before any model runs:

| # | Deviation | Why it exists | How it is handled |
| --- | --- | --- | --- |
| 1 | Ab1 is missing | The antibody benchmark is not in the public CodonBERT release | Reproduce four of five benchmarks; ask Mina on October 20 whether the file can be obtained |
| 2 | The 999-nucleotide length rule is inferred | The paper reports used counts, not preprocessing | Recovered from four counts at once; applied in one place, `blf_benchmark_rows` |
| 3 | mRNA stability leaks across its published split | 68% of test rows reuse a training sequence, with conflicting labels | Report the published split *and* the sequence-grouped split; treat the gap as a result |
| 4 | Sequences without a start codon need a protein stream | The paper does not state its treatment | Decided and recorded in step 3, with the chosen rule applied identically to every encoder set |

Two more deviations are expected but cannot be settled from the data alone: BioLangFusion
publishes no code we could verify, so the TextCNN layer sizes, the projection dimensions, and
the training seeds are reimplemented from the paper's description. Those are the second
question for the October 20 meeting.


---

## What step 2 needs from here

Step 2, `02_token_embeddings.ipynb`, extracts per-token embeddings for BioLangFusion's three
encoders on exactly the rows this notebook selected. It needs, for each benchmark:

- the rows `blf_benchmark_rows` returns, in file order, so an embedding matrix can be matched
  back to its labels and split by position;
- each row's sequence in the alphabet its encoder reads, which
  [`Dataset.inputs_for`](../shared_code/datasets.py) already supplies for the derived setting;
- the token budget: at most 999 nucleotides is 999 RNA tokens for RNA-FM, 333 codons for the
  protein stream, and 167 six-mer tokens for the DNA stream, all within every encoder's limit,
  so no sequence is truncated at embedding time.

Token-level embeddings are far larger than the pooled vectors of Stage 1: roughly 999 tokens
times 640 dimensions per sequence for RNA-FM alone. They belong on cluster scratch storage,
not in this repository, which is why `Code/*/embeddings/` is ignored by Git.
